In [1]:
!pip install chromadb sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.8/20.8 MB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 84.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.3/132.3 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.0/208.0 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 7.7 MB/s eta

In [3]:
# VectorDB에 텍스트 파일 저장 후 유사한 텍스트 읽기
import os, re, json, uuid    # uuid : 고유식별 id 생성용
from typing import List    # type hint 기능 제공 (가독성)
from sentence_transformers import SentenceTransformer    # 문장 단위의 의미 임베딩 라이브러리
from chromadb import PersistentClient

# 아래) 민감한 부분이기 때문에 따로 파일로 만들어서 불러오는 것이 좋다(보안)
TXT_PATH = "sample.json"
CHROMA_DIR = ".chroma_json_demo"
COLLECTION = "json_docs"
MODEL_NAME = "all-MiniLM-L6-v2"

model = SentenceTransformer(MODEL_NAME)
client = PersistentClient(path=CHROMA_DIR)
collection = client.get_or_create_collection(COLLECTION)

In [7]:
def upsert_jsonFunc(json_path:str):    # 저장 함수
  if not os.path.exists(json_path):
    raise FileNotFoundError("파일 없음")

  with open(json_path, "r", encoding="utf-8", errors="ignore") as f:
    data = json.load(f)
    if not data:
      print("자료 없음")
      return

  ids = [item.get("id", str(uuid.uuid4())) for item in data]    # 각 문단에 적용할 고유 id 생성
  docs = [f"{item.get('title', '')}, {item.get('content', '')}" for item in data]
  metas = [{"title": item.get('title', ''), "source":os.path.basename(json_path)} for item in data]
  embs = model.encode(docs, normalize_embeddings=True).tolist()
  # print(embs)

  # 저장
  collection.add(ids=ids, documents=docs, embeddings=embs, metadatas=metas)
  print(f"[저장 완료] : {len(data)}개 문단")    # [저장 완료] : 2개 문단


def searchFunc(query:str, k:int):    # 검색 함수
  q_emb = model.encode(query, normalize_embeddings=True).tolist()
  res = collection.query(query_embeddings=q_emb, n_results=k)

  docs = res.get('documents', [[]])[0]    # 예외 방지용 패턴
  metas = res.get('metadatas', [[]])[0]
  ids = res.get('ids', [[]])[0]
  dists = res.get('distances', [[]])[0] # Changed 'distance' to 'distances' based on common ChromaDB output

  for i, (doc, meta, _id, dist) in enumerate(zip(docs, metas, ids, dists)): # Renamed dists to dist
    print(f"\n[{i}] id={_id}")
    print(f"source={meta.get('source')}, dist={dist:.4f}") # Removed len={meta.get('len')} as it's not present in metas
    print(doc[:100] + ("..." if len(doc) > 100 else ""))

In [8]:
if __name__ == "__main__":
  upsert_jsonFunc(TXT_PATH)    # 저장 한 번 후 주석처리
  print("\n검색 예: ")
  searchFunc("노드와 포인터로 이루어진 자료구조 만세", k=3)

[저장 완료] : 4개 문단

검색 예: 

[0] id=p003
source=sample.json, dist=0.4880
정렬 알고리즘, 선택 정렬과 삽입 정렬은 단순하지만 느리고, 퀵 정렬은 평균적으로 빠른 성능을 보입니다.

[1] id=p002
source=sample.json, dist=0.6066
링크드 리스트, 링크드 리스트는 노드와 포인터로 이루어진 자료구조입니다. 삽입과 삭제가 효율적입니다.

[2] id=p001
source=sample.json, dist=0.6188
배열의 기본 개념, 배열은 같은 자료형의 데이터를 연속된 공간에 저장하는 자료구조입니다. 인덱스를 이용해 빠르게 접근할 수 있습니다.
